In [ ]:
"""
Chatbot de Cybersécurité - Version Optimisée avec Prompt Détaillé
Performances améliorées avec génération rapide
"""

# =============================
# Installation
# =============================
import subprocess, sys

def install_packages():
    packages = [
        'transformers', 'faiss-cpu', 'gradio', 'torch',
        'sentence-transformers', 'accelerate', 'bitsandbytes', 'tqdm'
    ]
    print("Installation des dépendances...")
    for package in packages:
        try:
            subprocess.check_call([sys.executable, '-m', 'pip', 'install', '-q', package])
        except:
            # Simple passage en cas d'erreur d'installation
            pass
    print("Installation terminée.\n")

install_packages()

# =============================
# Importations
# =============================
import json, torch, numpy as np, faiss, warnings
import gradio as gr
from tqdm import tqdm
from transformers import AutoTokenizer, AutoModelForCausalLM, BitsAndBytesConfig
from sentence_transformers import SentenceTransformer

warnings.filterwarnings('ignore')

# =============================
# CONFIGURATION OPTIMISÉE
# =============================
class Config:
    MODEL_NAME = "Qwen/Qwen2.5-1.5B-Instruct"
    EMBED_MODEL = "all-MiniLM-L6-v2"
    TOP_K_RESULTS = 3
    MAX_NEW_TOKENS = 200
    TEMPERATURE = 0.5
    TOP_P = 0.85
    DEVICE = "cuda" if torch.cuda.is_available() else "cpu"

    # Prompt système optimisé et détaillé
    SYSTEM_PROMPT = """Tu es Cyberbersecurity analyste et chasseur de menaces, un assistant expert en cybersécurité spécialisé dans l'analyse de menaces.

RÔLE ET EXPERTISE:
- Analyser des données de menaces cyber provenant de forums underground
- Identifier acteurs malveillants, outils de hacking, et vulnérabilités
- Évaluer le niveau de dangerosité (threat_score 1-10)
- Extraire insights pertinents sur tactiques, techniques et procédures (TTP)

FORMAT DES DONNÉES:
Chaque document contient: acteur, site source, type de menace, score, titre, contenu, entités (liens Telegram/Discord/GitHub, emails, mots-clés)

INSTRUCTIONS DE RÉPONSE:
1. CONCISION: Réponses directes, 2-4 phrases maximum
2. STRUCTURE: [Info principale] + [Détail technique] + [Recommandation si pertinent]
3. PRÉCISION: Citer acteurs, scores, sites sources
4. SYNTHÈSE: Si plusieurs documents, grouper par similarité
5. VOCABULAIRE: Termes techniques appropriés (exploit, C2, RAT, etc.)

EXEMPLES:
Q: "Qui est l'acteur X?"
R: "X est actif sur [site], score [N]/10. Partage [type outils/données]. Entités: [liens]."

Q: "Quelles menaces ransomware?"
R: "3 acteurs identifiés: [noms] sur [sites]. Scores 7-9/10. Ciblent [secteurs]. Recommandation: surveiller [indicateurs]."

Réponds UNIQUEMENT avec les infos du contexte fourni. Si absent, dis "Aucune donnée disponible"."""

# =============================
# CHATBOT OPTIMISÉ
# =============================
class CyberChatbot:
    def __init__(self):
        self.index = None
        self.texts = []
        self.raw_data = []
        self.embed_model = None
        self.llm_model = None
        self.tokenizer = None
        self.models_loaded = False
        self.data_loaded = False

    def combine_fields(self, entry):
        """Combine efficacement les champs JSON - Version optimisée"""
        if not isinstance(entry, dict):
            return str(entry)

        # Extraction rapide des champs essentiels
        parts = []

        # Champs prioritaires pour la recherche
        priority_fields = ['actor', 'threat_type', 'website', 'title', 'content_clean']
        for field in priority_fields:
            if field in entry and entry[field]:
                value = str(entry[field])[:500]  # Limite pour éviter textes trop longs
                parts.append(f"{field}:{value}")

        return " | ".join(parts) if parts else str(entry)

    def load_all(self, file_path):
        """Chargement automatique complet"""
        print("\n" + "="*60)
        print("INITIALISATION DU CHATBOT OPTIMISÉ")
        print("="*60 + "\n")

        # 1. JSON
        print("Étape 1/4: Lecture du fichier JSON...")
        try:
            with open(file_path, 'r', encoding='utf-8') as f:
                data = json.load(f)

            if not isinstance(data, list):
                print("Erreur: Format JSON invalide")
                return False

            self.raw_data = data
            print(f"-> {len(data):,} entrées chargées\n")
        except Exception as e:
            print(f"Erreur: {e}")
            return False

        # 2. LLM avec optimisations
        print("Étape 2/4: Chargement du modèle LLM (optimisé)...")
        try:
            quantization_config = BitsAndBytesConfig(
                load_in_8bit=True,
                bnb_8bit_compute_dtype=torch.float16
            )

            self.tokenizer = AutoTokenizer.from_pretrained(
                Config.MODEL_NAME,
                trust_remote_code=True
            )

            if self.tokenizer.pad_token is None:
                self.tokenizer.pad_token = self.tokenizer.eos_token

            self.llm_model = AutoModelForCausalLM.from_pretrained(
                Config.MODEL_NAME,
                quantization_config=quantization_config,
                device_map="auto",
                trust_remote_code=True,
                torch_dtype=torch.float16
            )

            self.llm_model.eval()

            # Optimisation: utiliser torch.compile si disponible (PyTorch 2.0+)
            try:
                if hasattr(torch, 'compile'):
                    print("   -> Optimisation avec torch.compile...")
                    self.llm_model = torch.compile(self.llm_model, mode="reduce-overhead")
            except:
                pass

            self.models_loaded = True
            print("-> Modèle LLM chargé et optimisé\n")
        except Exception as e:
            print(f"Erreur: {e}")
            return False

        # 3. Embeddings
        print("Étape 3/4: Chargement du modèle d'embeddings...")
        try:
            self.embed_model = SentenceTransformer(Config.EMBED_MODEL, device=Config.DEVICE)
            print("-> Modèle d'embeddings chargé\n")
        except Exception as e:
            print(f"Erreur: {e}")
            return False

        # 4. Indexation optimisée
        print("Étape 4/4: Indexation des documents...")
        try:
            self.texts = []
            embeddings_list = []

            # Utilisation directe des embeddings pré-calculés
            for item in tqdm(data, desc="Traitement", ncols=70):
                text = self.combine_fields(item)
                self.texts.append(text)

                # OPTIMISATION: Utiliser les embeddings existants
                if 'embedding' in item and item['embedding']:
                    embeddings_list.append(item['embedding'])
                else:
                    # Fallback si embedding absent
                    emb = self.embed_model.encode([text], convert_to_numpy=True)[0]
                    embeddings_list.append(emb.tolist())

            embeddings = np.array(embeddings_list, dtype='float32')

            # Index FAISS optimisé
            dimension = embeddings.shape[1]
            self.index = faiss.IndexFlatL2(dimension)
            self.index.add(embeddings)

            self.data_loaded = True
            print(f"\n-> {len(data):,} documents indexés avec succès\n")
        except Exception as e:
            print(f"Erreur: {e}")
            return False

        print("="*60)
        print("CHATBOT PRÊT - Mode haute performance activé")
        print("="*60 + "\n")
        return True

    def search_relevant_docs(self, question):
        """Recherche sémantique optimisée"""
        if not self.data_loaded:
            return []

        try:
            q_emb = self.embed_model.encode([question], convert_to_numpy=True).astype('float32')
            distances, indices = self.index.search(q_emb, Config.TOP_K_RESULTS)

            relevant_docs = []
            for idx, dist in zip(indices[0], distances[0]):
                # Seuil ajusté pour pertinence
                if dist < 2.5:
                    relevant_docs.append({
                        'text': self.texts[idx],
                        'data': self.raw_data[idx],
                        'distance': float(dist)
                    })

            return relevant_docs
        except Exception as e:
            print(f"Erreur recherche: {e}")
            return []

    def format_context_compact(self, relevant_docs):
        """Contexte ultra-compact pour génération rapide"""
        if not relevant_docs:
            return "Aucun document."

        context_parts = []

        for i, doc in enumerate(relevant_docs[:3], 1):  # Max 3 docs
            data = doc['data']

            # Format condensé
            doc_info = f"[Doc{i}]"

            if 'actor' in data:
                doc_info += f" Acteur:{data['actor']}"
            if 'threat_type' in data:
                doc_info += f" Type:{data['threat_type']}"
            if 'threat_score' in data:
                doc_info += f" Score:{data['threat_score']}/10"
            if 'website' in data:
                doc_info += f" Site:{data['website']}"
            if 'title' in data and data['title']:
                doc_info += f" Titre:{data['title'][:100]}"

            # Contenu condensé
            if 'content_clean' in data and data['content_clean']:
                content = data['content_clean'][:200].replace('\n', ' ')
                doc_info += f" Info:{content}"

            # Entités importantes
            if 'entities' in data:
                entities = data['entities']
                if isinstance(entities, dict):
                    if entities.get('telegram_links'):
                        doc_info += f" TG:{len(entities['telegram_links'])}"
                    if entities.get('discord_links'):
                        doc_info += f" DC:{len(entities['discord_links'])}"
                    if entities.get('threat_keywords'):
                        doc_info += f" Keywords:{','.join(entities['threat_keywords'][:3])}"

            context_parts.append(doc_info)

        return "\n".join(context_parts)

    def generate_llm_response(self, question, context):
        """Génération ultra-optimisée avec prompt système"""
        if not self.models_loaded:
            return "Modele non charge."

        try:
            # Prompt optimisé avec system prompt
            # Utilise le format de chat Qwen pour une meilleure performance
            prompt = f"""<|im_start|>system
{Config.SYSTEM_PROMPT}<|im_end|>
<|im_start|>user
Contexte:
{context}

Question: {question}<|im_end|>
<|im_start|>assistant
"""

            # Tokenization optimisée
            inputs = self.tokenizer(
                prompt,
                return_tensors="pt",
                truncation=True,
                max_length=1200  # Réduit pour rapidité
            ).to(self.llm_model.device)

            # Génération rapide avec paramètres optimisés
            with torch.no_grad():
                outputs = self.llm_model.generate(
                    **inputs,
                    max_new_tokens=Config.MAX_NEW_TOKENS,
                    temperature=Config.TEMPERATURE,
                    top_p=Config.TOP_P,
                    do_sample=True,
                    pad_token_id=self.tokenizer.pad_token_id,
                    eos_token_id=self.tokenizer.eos_token_id,
                    num_beams=1,  # Greedy pour vitesse
                    early_stopping=True,
                    repetition_penalty=1.15  # Évite répétitions
                )

            # Décodage
            response = self.tokenizer.decode(
                outputs[0][inputs['input_ids'].shape[1]:],
                skip_special_tokens=True
            ).strip()

            # Nettoyage
            if response.startswith('<|im_end|>'):
                response = response[11:].strip()

            if not response or len(response) < 10:
                return self.fallback_response(question, context)

            return response

        except Exception as e:
            print(f"[ERREUR] Generation: {e}")
            return self.fallback_response(question, context)

    def fallback_response(self, question, context):
        """Réponse directe et rapide"""
        lines = context.split('\n')

        if not lines or lines[0] == "Aucun document.":
            return "Aucune information trouvee dans la base.\n\nReformulez votre question."

        # Extraction rapide des infos clés
        response = f"Resultats pour '{question}':\n\n"

        for line in lines[:3]:
            if '[Doc' in line:
                # Parser le format condensé
                response += f"• {line[6:]}\n"

        response += "\nReponse extraite directement (mode fallback)"
        return response

    def chat(self, question):
        """Point d'entrée optimisé - Génération rapide"""
        if not question or not question.strip():
            return "Veuillez poser une question."

        if not self.data_loaded:
            return "Base de donnees non chargee."

        if not self.models_loaded:
            return "Mode extraction directe active.\n\n" + self.fallback_mode(question)

        try:
            # 1. Recherche rapide
            relevant_docs = self.search_relevant_docs(question)

            if not relevant_docs:
                return "Aucun document pertinent trouve.\n\nReformulez votre question ou essayez des mots-cles differents."

            # 2. Contexte compact
            context = self.format_context_compact(relevant_docs)

            # 3. Génération rapide
            response = self.generate_llm_response(question, context)

            return response

        except Exception as e:
            print(f"[ERREUR] {e}")
            return self.fallback_mode(question)

    def fallback_mode(self, question):
        """Mode secours rapide"""
        relevant_docs = self.search_relevant_docs(question)

        if not relevant_docs:
            return "Aucune information disponible."

        context = self.format_context_compact(relevant_docs)
        return self.fallback_response(question, context)

# =============================
# SÉLECTION FICHIER
# =============================
def get_json_file():
    """Sélection automatique du fichier JSON"""
    print("\n" + "="*60)
    print("CHATBOT CYBERSÉCURITÉ - VERSION OPTIMISÉE")
    print("="*60)
    print("\nSélection du fichier JSON...\n")

    # Essayer Google Colab
    try:
        from google.colab import files
        print("Veuillez sélectionner votre fichier JSON (par ex: data.json) :")
        uploaded = files.upload()
        if uploaded:
            filename = list(uploaded.keys())[0]
            print(f"\n-> Fichier: {filename}")
            return filename
    except ImportError:
        pass

    # Sinon tkinter
    try:
        import tkinter as tk
        from tkinter import filedialog

        root = tk.Tk()
        root.withdraw()
        file_path = filedialog.askopenfilename(
            title="Sélectionnez votre fichier JSON",
            filetypes=[("JSON files", "*.json"), ("All files", "*.*")]
        )
        root.destroy()

        if file_path:
            print(f"\n-> Fichier: {file_path}")
            return file_path
    except:
        pass

    # Si rien ne fonctionne
    print("\nAttention: Impossible d'ouvrir la boîte de dialogue de sélection de fichier.")
    return None

# =============================
# INTERFACE GRADIO
# =============================
def create_interface(bot):
    """Interface utilisateur améliorée"""
    def chat_fn(message, history):
        return bot.chat(message)

    # Statistiques
    total_docs = len(bot.raw_data)
    total_actors = len(set(item.get('actor', 'Unknown') for item in bot.raw_data))
    total_sites = len(set(item.get('website', 'Unknown') for item in bot.raw_data))

    # Calcul du score moyen de menace
    avg_threat = sum(item.get('threat_score', 0) for item in bot.raw_data) / total_docs if total_docs > 0 else 0

    with gr.Blocks(
        theme=gr.themes.Soft(primary_hue="red", secondary_hue="slate"),
        title="CyberGuard - Chatbot Cybersecurite",
        css="""
        .gradio-container {font-family: 'Inter', sans-serif;}
        .header-stats {background: linear-gradient(135deg, #667eea 0%, #764ba2 100%);
                      padding: 20px; border-radius: 10px; color: white; margin-bottom: 20px;}
        """
    ) as interface:

        gr.Markdown(f"""
        <div class="header-stats">

        # CyberGuard - Assistant d'Analyse de Menaces

        **Intelligence artificielle spécialisée en cybersécurité**

        **Base de données:** {total_docs:,} posts analysés | **{total_actors}** acteurs | **{total_sites}** sites sources | **Score moyen:** {avg_threat:.1f}/10

        **Mode Performance:** Réponses rapides avec prompt système optimisé

        </div>

        ---

        ### Posez vos questions sur les menaces cyber

        **CyberGuard** analyse en temps réel votre question et recherche dans **{total_docs:,}** documents de menaces pour vous fournir une réponse précise et contextuelle.
        """)

        gr.ChatInterface(
            fn=chat_fn,
            examples=[
                "Qui est l'acteur Tana et quel est son niveau de dangerosité ?",
                "Liste les 5 acteurs les plus dangereux par threat_score",
                "Quels types de ransomware sont présents dans la base ?",
                "Quels sites sont les plus utilisés pour diffuser des malwares ?",
                "Analyse les acteurs qui utilisent Telegram",
                "Qui cible GitHub dans cette base de données ?",
                "Explique les différents types de menaces identifiées",
                "Résume les principales tactiques des acteurs malveillants",
                "Y a-t-il des acteurs qui partagent des exploits zero-day ?",
                "Quels outils de hacking sont les plus distribués ?"
            ],
            chatbot=gr.Chatbot(
                height=550
            ),
            textbox=gr.Textbox(
                placeholder="Ex: Analyse les menaces de type ransomware distribuées sur cracked.to",
                container=False
            )
        )

        gr.Markdown("""
        ---

        ### À propos de CyberGuard

        **Technologies:**
        - **LLM:** Qwen2.5-1.5B-Instruct (quantization 8-bit)
        - **Recherche:** FAISS + Sentence-Transformers (all-MiniLM-L6-v2)
        - **Architecture:** RAG (Retrieval-Augmented Generation) optimisé
        - **Performance:** Prompt système détaillé + génération accélérée

        **Fonctionnalités:**
        - Analyse de menaces en temps réel
        - Identification d'acteurs malveillants
        - Évaluation de scores de dangerosité
        - Extraction d'entités (Telegram, Discord, GitHub, emails)
        - Classification par types de menaces

        **Développé par Nadia Kandoul** | 2024 | Mode Haute Performance
        """)

    return interface

# =============================
# MAIN
# =============================
if __name__ == "__main__":
    # Sélection du fichier
    json_file = get_json_file()

    if not json_file:
        print("\nOperation annulee.")
        sys.exit(1)

    # Création et chargement automatique
    bot = CyberChatbot()
    success = bot.load_all(json_file)

    if not success:
        print("\nEchec du chargement.")
        sys.exit(1)

    # Lancement interface - CORRECTION APPLIQUÉE ICI
    print("\nLancement de l'interface Gradio...\n")
    interface = create_interface(bot)
    interface.launch(
        share=True,
        show_error=True
        # Retrait de server_name="0.0.0.0" et server_port=7860 pour résoudre l'OSError
    )

Installation des dépendances...
Installation terminée.


CHATBOT CYBERSÉCURITÉ - VERSION OPTIMISÉE

Sélection du fichier JSON...

Veuillez sélectionner votre fichier JSON (par ex: data.json) :


Saving cti_enriched_dataset.json to cti_enriched_dataset.json

-> Fichier: cti_enriched_dataset.json

INITIALISATION DU CHATBOT OPTIMISÉ

Étape 1/4: Lecture du fichier JSON...
-> 22,622 entrées chargées

Étape 2/4: Chargement du modèle LLM (optimisé)...


tokenizer_config.json: 0.00B [00:00, ?B/s]

vocab.json: 0.00B [00:00, ?B/s]

merges.txt: 0.00B [00:00, ?B/s]

...

config.json:   0%|          | 0.00/660 [00:00<?, ?B/s]

`torch_dtype` is deprecated! Use `dtype` instead!


model.safetensors:   0%|          | 0.00/3.09G [00:00<?, ?B/s]

Erreur: Can't load the model for 'Qwen/Qwen2.5-1.5B-Instruct'. If you were trying to load it from 'https://huggingface.co/models', make sure you don't have a local directory with the same name. Otherwise, make sure 'Qwen/Qwen2.5-1.5B-Instruct' is the correct path to a directory containing a file named pytorch_model.bin, tf_model.h5, model.ckpt or flax_model.msgpack.

Echec du chargement.


SystemExit: 1